In [1]:
import os
# os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"
# os.environ["CUDA_VISIBLE_DEVICES"]="3"

import datasets
import json
import numpy as np
import pandas as pd
from collections import defaultdict

/home/artem.shelmanov/miniconda3/envs/dllm/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from llada.generate import *
from llada.load_llada import *

In [3]:
model = load_model('GSAI-ML/LLaDA-1.5', device_map="auto")
tokenizer = load_tokenizer('GSAI-ML/LLaDA-1.5')

Loading weights: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 291/291 [00:10<00:00, 29.01it/s]


In [4]:
prompt = "How old is Donald Trump? Answer as briefly as possible"
device = "cuda"

# Add special tokens for the Instruct model. The Base model does not require the following two lines.
m = [{"role": "user", "content": prompt}, ]
prompt = tokenizer.apply_chat_template(m, add_generation_prompt=True, tokenize=False)

input_ids = tokenizer(prompt)['input_ids']
input_ids = torch.tensor(input_ids).to(device).unsqueeze(0)

In [5]:
out = generate_with_dual_cache(model, tokenizer, input_ids, steps=128, gen_length=128, block_length=32, temperature=0., remasking='low_confidence', threshold=0.9, factor=1)

[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


In [6]:
out[-1]["generation_trace"]

[['Donald Trump was born on6 66, '],
 ['Donald Trump was born on6  old6 '],
 ['Donald Trump is  on6 years old.'],
 ['Donald Trump is 76 years old.'],
 ['Donald Trump is 7  years old.'],
 ['Donald Trump is 72 years old.'],
 ['Donald Trump is 70 years old.'],
 ['Donald Trump is 70 years old.']]

In [12]:
import torch.nn.functional as F
import torch

input1 = torch.randn(100, 4096)
input2 = torch.randn(100, 4096)
output1 = F.cosine_similarity(input1, input2)
print(output1)

tensor([-0.0012,  0.0089,  0.0311, -0.0266, -0.0302, -0.0271, -0.0050,  0.0033,
         0.0142,  0.0281, -0.0030, -0.0119,  0.0055, -0.0328,  0.0099,  0.0143,
        -0.0168, -0.0411,  0.0238,  0.0020,  0.0050, -0.0268, -0.0174,  0.0161,
         0.0180,  0.0203,  0.0012,  0.0216, -0.0042, -0.0102,  0.0017, -0.0005,
        -0.0082,  0.0054,  0.0110,  0.0030, -0.0280,  0.0037, -0.0204,  0.0131,
        -0.0036,  0.0021, -0.0114,  0.0131,  0.0071, -0.0115, -0.0078, -0.0153,
        -0.0027,  0.0025,  0.0298, -0.0106, -0.0262,  0.0325,  0.0092,  0.0015,
        -0.0069, -0.0097,  0.0072, -0.0178,  0.0173, -0.0054,  0.0025,  0.0150,
         0.0378,  0.0189,  0.0210, -0.0130, -0.0158, -0.0031, -0.0042, -0.0010,
        -0.0108,  0.0251,  0.0060, -0.0308,  0.0054,  0.0278,  0.0053, -0.0236,
        -0.0139,  0.0010, -0.0087,  0.0089, -0.0287,  0.0341,  0.0270, -0.0230,
        -0.0060, -0.0159,  0.0064, -0.0064, -0.0147,  0.0301,  0.0204, -0.0079,
         0.0173, -0.0212,  0.0232,  0.00

In [18]:
output2 = F.cosine_similarity(input1[:1], input2)
print(output2)

tensor([-0.0012, -0.0177, -0.0094, -0.0113, -0.0093,  0.0041,  0.0150,  0.0035,
        -0.0322, -0.0104, -0.0022, -0.0220, -0.0097,  0.0190,  0.0240, -0.0032,
        -0.0170, -0.0039,  0.0196, -0.0009,  0.0377,  0.0002,  0.0121,  0.0003,
         0.0107,  0.0095,  0.0451, -0.0143,  0.0026, -0.0073, -0.0039,  0.0168,
         0.0075,  0.0133,  0.0098, -0.0033, -0.0080,  0.0028,  0.0114, -0.0413,
        -0.0278,  0.0043,  0.0022, -0.0130, -0.0375, -0.0103, -0.0092, -0.0047,
        -0.0309, -0.0167, -0.0059, -0.0058, -0.0199, -0.0048,  0.0255,  0.0056,
        -0.0037,  0.0147,  0.0038,  0.0191, -0.0020, -0.0250, -0.0055,  0.0194,
        -0.0135,  0.0047, -0.0230, -0.0014,  0.0031, -0.0256, -0.0179,  0.0011,
         0.0135, -0.0174, -0.0082, -0.0015, -0.0173,  0.0151, -0.0275,  0.0236,
         0.0092,  0.0162, -0.0162,  0.0114,  0.0036,  0.0044, -0.0059,  0.0026,
         0.0172,  0.0356, -0.0113,  0.0041,  0.0060, -0.0059, -0.0287,  0.0036,
        -0.0108,  0.0002, -0.0048,  0.00

In [19]:
F.cosine_similarity(input1[:1], input2[-1])


tensor([0.0039])

In [26]:
(input1[0] * input2[-1]).sum() / (input1[0].norm() * input2[-1].norm())

tensor(0.0039)